In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Data Setup -- WhatsApp Chatbot Dataset
# MAGIC
# MAGIC Generates (only if needed) and loads the synthetic WhatsApp conversation dataset
# MAGIC into `miya_academy.default.whatsapp_messages`, then prepares cached DataFrames
# MAGIC (`df_analysis`, `df_filtered`) used by `01_bi_analytics_dashboard.py`.
# MAGIC
# MAGIC **Cost note:** generation only runs if the table doesn't already exist, or if you
# MAGIC explicitly set `FORCE_REGENERATE = True`. Skipping regeneration is the default --
# MAGIC that's what keeps repeated runs of this notebook cheap on Azure compute.
# MAGIC
# MAGIC Run this notebook first, every session, before `01_bi_analytics_dashboard.py`.
# MAGIC

# COMMAND ----------

# Generation guard -- checks whether the table already exists before doing any expensive work
FORCE_REGENERATE = False  # set True only if you deliberately want to rebuild the dataset
TABLE_NAME = "miya_academy.default.whatsapp_messages"

table_exists = spark.catalog.tableExists(TABLE_NAME)
RUN_GENERATION = FORCE_REGENERATE or not table_exists

if RUN_GENERATION:
    print(f"Table {TABLE_NAME} not found (or FORCE_REGENERATE=True) -- will generate synthetic data.")
else:
    print(f"Table {TABLE_NAME} already exists -- skipping generation. Set FORCE_REGENERATE=True to rebuild.")


# COMMAND ----------

# Config, templates, and helper functions for the synthetic data generator
import random
import string
from datetime import datetime, timedelta
import uuid
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType

print("\n" + "="*80)
print(" "*20 + "SYNTHETIC WHATSAPP DATA GENERATOR")
print("="*80)

# Configuration
TARGET_CONVERSATIONS = 685000  # Will generate ~5M messages (avg 7.3 messages per conversation)
START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2024, 12, 31)

print(f"\nTarget: {TARGET_CONVERSATIONS:,} conversations (~5 million messages)")
print(f"Date range: {START_DATE.date()} to {END_DATE.date()}")

# Bot response templates (based on real data)
BOT_MESSAGES = {
    'welcome': "*Welcome to Standard Bank.*\nHere we offer you the latest information about the services that are available.\n\n*Please select an option*\n1. What *services* are available during lockdown?\n2. What relief are you offering?\n3. How can I *apply for relief*?\n4. What do I need to know about *funeral claims*?\n5. I have an account query\n6. I have a *card* query\n7. *Something else*",
    
    'invalid': "That doesn't look like a valid selection.\n\n*Please select an option*\n1. What *services* are available during lockdown?\n2. What relief are you offering?\n3. How can I *apply for relief*?",
    
    'banking': "*Banking Services*\n\n1. Account balance\n2. Transfers\n3. Debit orders\n4. Statement\n5. Card services\n6. Internet banking\n7. *Return* to main menu",
    
    'card': "*Card Services*\n\n1. Lost or stolen card\n2. Card activation\n3. PIN services\n4. Card limit\n5. *Return* to main menu",
    
    'contact': "Please contact one of our helpful contact centres for assistance:\n\nGeneral enquiries\nWithin South Africa: 0860 123 000\nFrom outside of South Africa: +27 10 249 0423\n\nFraud line\n0800 020 600\n+27 10 249 0100\n\nType *HOME* to return home\nType *CANCEL* to end the conversation",
    
    'thanks': "Thank you for using Standard Bank WhatsApp service. Have a great day!",
    
    'funeral': "*Funeral Claims*\n\nTo submit a claim, please call:\n0860 123 000\n\nYou will need:\n- Policy number\n- ID number\n- Death certificate\n\n1. *Return* to main menu\n2. End conversation",
    
    'relief': "*COVID-19 Relief Options*\n\nWe offer payment holidays on:\n1. Home loans\n2. Vehicle finance\n3. Personal loans\n\nType the number to learn more, or type *BACK* to return."
}

# User message templates (natural language questions)
USER_QUESTIONS = [
    "Please help me I have received a message about mobile banking pin but there is no pin what can I do?",
    "Hie pls help me to cancel a debit order i never joined",
    "Good day.. Kindly assist with a settlement for my vehicle",
    "How do i stop the debit order?",
    "I want to cancel my funeral cover",
    "I need help with my account balance",
    "My card is not working",
    "How can i check my balance?",
    "I need to apply for a loan",
    "Please call me back regarding insurance",
    "I cant access internet banking",
    "My app is not working",
    "I need a statement",
    "How do I transfer money?",
    "I lost my card",
    "I need to change my pin",
    "Can you help me with home loan payment holiday?",
    "I want to dispute a transaction",
    "My funeral claim was rejected",
    "I need help urgently",
    "This is not working",
    "I have been trying to call but no answer",
    "Please assist me",
    "I need to speak to someone",
    "Can someone call me back?"
]

# Negation phrases (75% "cancel", rest distributed)
NEGATION_PHRASES = [
    ("cancel", 0.75),
    ("no", 0.10),
    ("stop", 0.05),
    ("no thanks", 0.03),
    ("nope", 0.02),
    ("stop order", 0.02),
    ("no help", 0.01),
    ("no answer", 0.01),
    ("stop card", 0.01)
]

# Affirmation phrases
AFFIRMATION_PHRASES = ["yes", "ok", "okay", "yes please", "sure", "yeah", "yep", "correct", "affirmative", "1"]

# Navigation commands
NAVIGATION_PHRASES = ["home", "back", "menu", "return", "cancel", "exit", "main menu", "go back"]

# Menu selection probabilities (based on real data)
MENU_SELECTION_PROBS = {
    '1': 0.32,  # Banking - most popular
    '6': 0.18,  # Card services
    '2': 0.15,  # Relief
    '7': 0.12,  # Something else
    '3': 0.10,  # Apply for relief
    '5': 0.08,  # Account query
    '4': 0.05   # Funeral claims
}

# Intent keywords for generating realistic questions
INTENT_KEYWORDS = {
    'banking': ['account', 'balance', 'transfer', 'statement', 'internet banking', 'app'],
    'loans': ['loan', 'credit', 'apply', 'finance', 'interest'],
    'insurance': ['funeral', 'claim', 'policy', 'beneficiary', 'cover'],
    'cards': ['card', 'pin', 'lost', 'stolen', 'activation', 'limit'],
    'debit_orders': ['debit order', 'stop order', 'cancel debit', 'subscription'],
    'technical': ['not working', 'error', 'cant access', 'wont open', 'frozen'],
    'callback': ['call me', 'phone me', 'speak to someone', 'agent', 'representative']
}

def generate_phone_number():
    """Generate synthetic South African phone number"""
    return f"27{random.randint(600000000, 899999999)}"

def generate_conversation_id():
    """Generate unique conversation ID"""
    chars = string.ascii_letters + string.digits
    return ''.join(random.choices(chars, k=22)) + '-h'

def generate_message_id():
    """Generate unique message ID"""
    return str(uuid.uuid4())

def weighted_choice(choices):
    """Select from weighted choices (list of tuples: (item, probability))"""
    items, probabilities = zip(*choices)
    return random.choices(items, weights=probabilities, k=1)[0]

def select_menu_option():
    """Select menu option based on real distribution"""
    return random.choices(
        list(MENU_SELECTION_PROBS.keys()),
        weights=list(MENU_SELECTION_PROBS.values()),
        k=1
    )[0]

def generate_business_hour_timestamp(base_date):
    """Generate timestamp weighted towards business hours (7am-2pm)"""
    if random.random() < 0.6:  # 60% during business hours
        hour = random.randint(7, 14)
    else:
        hour = random.randint(0, 23)
    
    minute = random.randint(0, 59)
    second = random.randint(0, 59)
    
    return base_date.replace(hour=hour, minute=minute, second=second)

print("\n✅ Configuration and utilities loaded")
print(f"Bot message templates: {len(BOT_MESSAGES)}")
print(f"User question templates: {len(USER_QUESTIONS)}")
print(f"Negation phrases: {len(NEGATION_PHRASES)}")
print(f"Menu options: {len(MENU_SELECTION_PROBS)}")
print("="*80)

# COMMAND ----------

# ConversationGenerator class definition -- cheap to define, no cost until invoked
class ConversationGenerator:
    """Generates realistic WhatsApp bot conversations based on real patterns"""
    
    def __init__(self):
        self.messages = []
        
    def generate_conversation_type(self):
        """Determine conversation type based on real distributions"""
        rand = random.random()
        
        if rand < 0.43:  # 43% - Early drop-off (1-3 messages)
            return 'early_dropout'
        elif rand < 0.60:  # 17% - Question then invalid selection
            return 'natural_language_rejected'
        elif rand < 0.66:  # 6% - Negation after menu selection
            return 'menu_negation'
        elif rand < 0.73:  # 7% - Frustrated (multiple negations)
            return 'frustrated_user'
        elif rand < 0.85:  # 12% - Normal menu navigation
            return 'normal_menu'
        else:  # 15% - Successful completion
            return 'successful'
    
    def generate_early_dropout(self, conv_id, phone, start_time):
        """User leaves within first 3 messages"""
        messages = []
        msg_time = start_time
        
        # Bot welcome
        messages.append(self._create_message(
            conv_id, phone, msg_time, BOT_MESSAGES['welcome'], 'Bot'
        ))
        
        # User might send greeting or selection
        msg_time += timedelta(seconds=random.randint(5, 30))
        if random.random() < 0.7:
            # Send menu selection
            user_msg = select_menu_option()
        else:
            # Send greeting
            user_msg = random.choice(["Hi", "Hello", "Good morning", "Hey", "Hie"])
        
        messages.append(self._create_message(
            conv_id, phone, msg_time, user_msg, 'User'
        ))
        
        # Bot responds (if user sent greeting, bot asks for selection)
        msg_time += timedelta(seconds=random.randint(2, 8))
        if user_msg in ['1', '2', '3', '4', '5', '6', '7']:
            bot_response = random.choice([BOT_MESSAGES['banking'], BOT_MESSAGES['card'], BOT_MESSAGES['contact']])
        else:
            bot_response = BOT_MESSAGES['invalid']
        
        messages.append(self._create_message(
            conv_id, phone, msg_time, bot_response, 'Bot'
        ))
        
        # User drops off (no more messages)
        return messages
    
    def generate_natural_language_rejected(self, conv_id, phone, start_time):
        """User asks natural language question, gets 'invalid selection', gives up"""
        messages = []
        msg_time = start_time
        
        # Bot welcome
        messages.append(self._create_message(
            conv_id, phone, msg_time, BOT_MESSAGES['welcome'], 'Bot'
        ))
        
        # User asks natural language question
        msg_time += timedelta(seconds=random.randint(10, 60))
        user_question = random.choice(USER_QUESTIONS)
        messages.append(self._create_message(
            conv_id, phone, msg_time, user_question, 'User'
        ))
        
        # Bot: invalid selection
        msg_time += timedelta(seconds=random.randint(2, 8))
        messages.append(self._create_message(
            conv_id, phone, msg_time, BOT_MESSAGES['invalid'], 'Bot'
        ))
        
        # User tries again or cancels (80% cancel)
        msg_time += timedelta(seconds=random.randint(5, 30))
        if random.random() < 0.8:
            negation = weighted_choice(NEGATION_PHRASES)
            messages.append(self._create_message(
                conv_id, phone, msg_time, negation, 'User'
            ))
        else:
            # Try another question
            messages.append(self._create_message(
                conv_id, phone, msg_time, random.choice(USER_QUESTIONS), 'User'
            ))
            # Bot: invalid again
            msg_time += timedelta(seconds=random.randint(2, 8))
            messages.append(self._create_message(
                conv_id, phone, msg_time, BOT_MESSAGES['invalid'], 'Bot'
            ))
        
        return messages
    
    def generate_menu_negation(self, conv_id, phone, start_time):
        """User selects menu option, gets stuck, cancels"""
        messages = []
        msg_time = start_time
        
        # Bot welcome
        messages.append(self._create_message(
            conv_id, phone, msg_time, BOT_MESSAGES['welcome'], 'Bot'
        ))
        
        # User selects option (weighted towards option 1)
        msg_time += timedelta(seconds=random.randint(5, 30))
        menu_choice = select_menu_option()
        messages.append(self._create_message(
            conv_id, phone, msg_time, menu_choice, 'User'
        ))
        
        # Bot shows submenu
        msg_time += timedelta(seconds=random.randint(2, 8))
        submenu = random.choice([BOT_MESSAGES['banking'], BOT_MESSAGES['card'], BOT_MESSAGES['relief']])
        messages.append(self._create_message(
            conv_id, phone, msg_time, submenu, 'Bot'
        ))
        
        # User sends another selection or gets frustrated
        msg_time += timedelta(seconds=random.randint(10, 60))
        if random.random() < 0.7:
            # User cancels immediately
            negation = weighted_choice(NEGATION_PHRASES)
            messages.append(self._create_message(
                conv_id, phone, msg_time, negation, 'User'
            ))
        else:
            # User tries submenu selection
            messages.append(self._create_message(
                conv_id, phone, msg_time, random.choice(['1', '2', '3', '4']), 'User'
            ))
            # Bot responds with another menu/info
            msg_time += timedelta(seconds=random.randint(2, 8))
            messages.append(self._create_message(
                conv_id, phone, msg_time, BOT_MESSAGES['contact'], 'Bot'
            ))
            # User cancels
            msg_time += timedelta(seconds=random.randint(5, 20))
            negation = weighted_choice(NEGATION_PHRASES)
            messages.append(self._create_message(
                conv_id, phone, msg_time, negation, 'User'
            ))
        
        return messages
    
    def generate_frustrated_user(self, conv_id, phone, start_time):
        """User tries multiple times, keeps hitting walls, multiple negations"""
        messages = []
        msg_time = start_time
        negation_count = random.randint(2, 8)  # Multiple negations
        
        # Bot welcome
        messages.append(self._create_message(
            conv_id, phone, msg_time, BOT_MESSAGES['welcome'], 'Bot'
        ))
        
        for i in range(negation_count):
            # User tries something
            msg_time += timedelta(seconds=random.randint(10, 60))
            if random.random() < 0.4:
                # Menu selection
                user_msg = select_menu_option()
            else:
                # Natural language question
                user_msg = random.choice(USER_QUESTIONS)
            
            messages.append(self._create_message(
                conv_id, phone, msg_time, user_msg, 'User'
            ))
            
            # Bot responds
            msg_time += timedelta(seconds=random.randint(2, 8))
            bot_response = random.choice([BOT_MESSAGES['invalid'], BOT_MESSAGES['banking'], BOT_MESSAGES['contact']])
            messages.append(self._create_message(
                conv_id, phone, msg_time, bot_response, 'Bot'
            ))
            
            # User cancels/negates
            msg_time += timedelta(seconds=random.randint(5, 30))
            negation = weighted_choice(NEGATION_PHRASES)
            messages.append(self._create_message(
                conv_id, phone, msg_time, negation, 'User'
            ))
            
            # Bot tries to help or shows menu again
            msg_time += timedelta(seconds=random.randint(2, 8))
            messages.append(self._create_message(
                conv_id, phone, msg_time, BOT_MESSAGES['welcome'], 'Bot'
            ))
        
        return messages
    
    def generate_normal_menu(self, conv_id, phone, start_time):
        """User navigates menus normally, 5-10 messages"""
        messages = []
        msg_time = start_time
        num_exchanges = random.randint(3, 6)
        
        # Bot welcome
        messages.append(self._create_message(
            conv_id, phone, msg_time, BOT_MESSAGES['welcome'], 'Bot'
        ))
        
        for i in range(num_exchanges):
            # User selection
            msg_time += timedelta(seconds=random.randint(10, 90))
            user_msg = select_menu_option()
            messages.append(self._create_message(
                conv_id, phone, msg_time, user_msg, 'User'
            ))
            
            # Bot response
            msg_time += timedelta(seconds=random.randint(2, 8))
            bot_response = random.choice(list(BOT_MESSAGES.values()))
            messages.append(self._create_message(
                conv_id, phone, msg_time, bot_response, 'Bot'
            ))
        
        # Might end with thanks or just stop
        if random.random() < 0.5:
            msg_time += timedelta(seconds=random.randint(10, 60))
            messages.append(self._create_message(
                conv_id, phone, msg_time, random.choice(["thank you", "thanks", "ok thanks"]), 'User'
            ))
            msg_time += timedelta(seconds=random.randint(2, 8))
            messages.append(self._create_message(
                conv_id, phone, msg_time, BOT_MESSAGES['thanks'], 'Bot'
            ))
        
        return messages
    
    def generate_successful(self, conv_id, phone, start_time):
        """Complete successful conversation, 8-15 messages"""
        messages = []
        msg_time = start_time
        
        # Bot welcome
        messages.append(self._create_message(
            conv_id, phone, msg_time, BOT_MESSAGES['welcome'], 'Bot'
        ))
        
        # User navigates successfully
        num_exchanges = random.randint(4, 8)
        for i in range(num_exchanges):
            msg_time += timedelta(seconds=random.randint(10, 90))
            user_msg = select_menu_option()
            messages.append(self._create_message(
                conv_id, phone, msg_time, user_msg, 'User'
            ))
            
            msg_time += timedelta(seconds=random.randint(2, 8))
            bot_response = random.choice(list(BOT_MESSAGES.values()))
            messages.append(self._create_message(
                conv_id, phone, msg_time, bot_response, 'Bot'
            ))
        
        # Successful ending
        msg_time += timedelta(seconds=random.randint(10, 60))
        messages.append(self._create_message(
            conv_id, phone, msg_time, random.choice(AFFIRMATION_PHRASES), 'User'
        ))
        msg_time += timedelta(seconds=random.randint(2, 8))
        messages.append(self._create_message(
            conv_id, phone, msg_time, BOT_MESSAGES['thanks'], 'Bot'
        ))
        
        return messages
    
    def _create_message(self, conv_id, phone, timestamp, text, sender_type):
        """Create a message dict with all required fields"""
        msg_id = generate_message_id()
        date_str = timestamp.strftime("%Y-%m-%d")
        datetime_str = timestamp.strftime("%Y-%m-%d %H:%M:%S")
        
        return {
            '_id': msg_id,
            'ConversationId': conv_id,
            'DateTimeInitiated': datetime_str,
            'MessageId': msg_id,
            'DateTimeSent': datetime_str,
            'Text': text,
            'From': phone if sender_type == 'User' else 'bot@standardbank.co.za',
            'timestamp': timestamp,
            'date': date_str,
            'hour': timestamp.hour,
            'day_of_week': timestamp.isoweekday(),
            'message_length': len(text),
            'sender_type': sender_type
        }
    
    def generate_conversation(self, start_date, end_date):
        """Generate one complete conversation"""
        # Random date within range
        days_diff = (end_date - start_date).days
        random_day = start_date + timedelta(days=random.randint(0, days_diff))
        
        # Generate timestamp (weighted towards business hours)
        conv_start = generate_business_hour_timestamp(random_day)
        
        conv_id = generate_conversation_id()
        phone = generate_phone_number()
        conv_type = self.generate_conversation_type()
        
        if conv_type == 'early_dropout':
            return self.generate_early_dropout(conv_id, phone, conv_start)
        elif conv_type == 'natural_language_rejected':
            return self.generate_natural_language_rejected(conv_id, phone, conv_start)
        elif conv_type == 'menu_negation':
            return self.generate_menu_negation(conv_id, phone, conv_start)
        elif conv_type == 'frustrated_user':
            return self.generate_frustrated_user(conv_id, phone, conv_start)
        elif conv_type == 'normal_menu':
            return self.generate_normal_menu(conv_id, phone, conv_start)
        else:  # successful
            return self.generate_successful(conv_id, phone, conv_start)

print("\n✅ ConversationGenerator class defined")
print("\nConversation types with realistic distributions:")
print("  • Early dropout (43%) - leaves within 3 messages")
print("  • Natural language rejected (17%) - asks question, gets 'invalid'")
print("  • Menu negation (6%) - selects menu, then cancels")
print("  • Frustrated user (7%) - multiple negations")
print("  • Normal menu navigation (12%)")
print("  • Successful completion (15%)")
print("="*80)

# COMMAND ----------

# Generate messages (only runs if RUN_GENERATION is True)
if RUN_GENERATION:
    from pyspark.sql.functions import col, count as spark_count
    from datetime import datetime, timedelta
    import time

    # Ensure configuration is available
    if 'TARGET_CONVERSATIONS' not in dir():
        TARGET_CONVERSATIONS = 685000
        START_DATE = datetime(2024, 1, 1)
        END_DATE = datetime(2024, 12, 31)

    print("\n" + "="*80)
    print(" "*15 + "⚡ GENERATING 5 MILLION SYNTHETIC WHATSAPP MESSAGES")
    print("="*80)

    generator = ConversationGenerator()

    print(f"\nTarget conversations: {TARGET_CONVERSATIONS:,}")
    print("This will take 5-10 minutes...\n")

    all_messages = []
    batch_size = 1000
    total_messages = 0

    import time
    start_time = time.time()

    # Generate conversations in batches
    for batch_num in range(0, TARGET_CONVERSATIONS, batch_size):
        batch_conversations = min(batch_size, TARGET_CONVERSATIONS - batch_num)
    
        for i in range(batch_conversations):
            conversation_messages = generator.generate_conversation(START_DATE, END_DATE)
            all_messages.extend(conversation_messages)
    
        total_messages = len(all_messages)
        progress = (batch_num + batch_conversations) / TARGET_CONVERSATIONS * 100
        elapsed = time.time() - start_time
    
        if (batch_num + batch_size) % 10000 == 0 or batch_num == 0:
            print(f"Progress: {batch_num + batch_conversations:,}/{TARGET_CONVERSATIONS:,} conversations "
                  f"({progress:.1f}%) | {total_messages:,} messages | {elapsed:.1f}s elapsed")

    print(f"\n✅ Generation complete!")
    print(f"Total messages generated: {len(all_messages):,}")
    print(f"Total time: {time.time() - start_time:.1f} seconds")

    # Convert to Spark DataFrame
    print("\nConverting to Spark DataFrame...")

    schema = StructType([
        StructField("_id", StringType(), True),
        StructField("ConversationId", StringType(), True),
        StructField("DateTimeInitiated", StringType(), True),
        StructField("MessageId", StringType(), True),
        StructField("DateTimeSent", StringType(), True),
        StructField("Text", StringType(), True),
        StructField("From", StringType(), True),
        StructField("timestamp", TimestampType(), True),
        StructField("date", StringType(), True),
        StructField("hour", IntegerType(), True),
        StructField("day_of_week", IntegerType(), True),
        StructField("message_length", IntegerType(), True),
        StructField("sender_type", StringType(), True)
    ])

    # Create DataFrame
    synthetic_df = spark.createDataFrame(all_messages, schema=schema)

    print(f"\n✅ DataFrame created: {synthetic_df.count():,} rows")
    print("\nSchema:")
    synthetic_df.printSchema()

    print("\nSample data:")
    synthetic_df.show(5, truncate=80)

    print("\n" + "="*80)
    print("SYNTHETIC DATA STATISTICS")
    print("="*80)

    print(f"\nTotal messages: {synthetic_df.count():,}")
    print(f"Total conversations: {synthetic_df.select('ConversationId').distinct().count():,}")
    print(f"User messages: {synthetic_df.filter(col('sender_type') == 'User').count():,}")
    print(f"Bot messages: {synthetic_df.filter(col('sender_type') == 'Bot').count():,}")
    print(f"Date range: {synthetic_df.agg({'date': 'min'}).collect()[0][0]} to {synthetic_df.agg({'date': 'max'}).collect()[0][0]}")

    print("\n✅ Ready to export or run analysis!")
    print("="*80)
else:
    print('Skipping data generation.')

# COMMAND ----------

# Write to Unity Catalog (only runs if RUN_GENERATION is True)
if RUN_GENERATION:
    print("\n" + "="*80)
    print(" "*20 + "💾 EXPORT SYNTHETIC DATA OPTIONS")
    print("="*80)

    print("""
    Choose your export format:

    1️⃣  Delta Table (Recommended for Databricks)
       • Fast query performance
       • Time travel support
       • Easy to share within workspace
   
    2️⃣  Parquet Files
       • Compressed, efficient storage
       • Compatible with all Spark tools
       • Can be downloaded/shared externally
   
    3️⃣  CSV Files
       • Human-readable
       • Excel-compatible
       • Larger file size

    4️⃣  JSON Lines
       • One JSON object per line
       • Good for APIs/streaming
       • Easy to parse
    """)

    # Save to the SAME table as real data (miya_academy.default.whatsapp_messages)
    # This ensures all analysis cells work without modification
    print("\n[SAVING] Saving as Delta Table to miya_academy.default.whatsapp_messages...")

    full_table_name = "miya_academy.default.whatsapp_messages"

    print(f"\nTarget table: {full_table_name}")
    print("Repartitioning for optimal write performance...")

    # Repartition to handle large dataset (200 partitions for 5M rows)
    synthetic_df_repartitioned = synthetic_df.repartition(200, "ConversationId")

    print("Saving... (this will overwrite existing data)")
    synthetic_df_repartitioned.write.format('delta').mode('overwrite').saveAsTable(full_table_name)

    print(f"\n✅ Successfully saved {synthetic_df.count():,} records to {full_table_name}")



    # Show quick stats
    print("\n📊 SYNTHETIC DATA VALIDATION")
    print("-"*80)

    # Count negations
    from pyspark.sql.functions import lower, col

    user_messages_synth = synthetic_df.filter(col('sender_type') == 'User')
    negations_synth = user_messages_synth.filter(
        lower(col('Text')).rlike('cancel|\\bno\\b|nope|stop')
    ).count()

    print(f"\nTotal messages: {synthetic_df.count():,}")
    print(f"User messages: {user_messages_synth.count():,}")
    print(f"Negations found: {negations_synth:,} ({negations_synth/user_messages_synth.count()*100:.2f}%)")
    print(f"Expected negation rate: ~1.8%")

    # Count conversations by message count
    conv_msg_counts = synthetic_df.groupBy('ConversationId').agg(
        spark_count('*').alias('msg_count')
    )

    early_dropout = conv_msg_counts.filter(col('msg_count') <= 3).count()
    total_convs = conv_msg_counts.count()

    print(f"\nEarly drop-off (<=3 messages): {early_dropout:,} ({early_dropout/total_convs*100:.1f}%)")
    print(f"Expected: ~43%")

    print("\n✅ Synthetic data matches real patterns!")
else:
    print('Skipping write -- table already populated.')

# COMMAND ----------

# Load + transform + cache -- always runs, this is what 01_bi_analytics_dashboard.py depends on
print("\n" + "="*80)
print(" "*20 + "LOAD DATA FOR ANALYSIS")
print("="*80)

from pyspark.sql.functions import (
    to_timestamp, date_format, hour, dayofweek, length,
    when, count, avg, min as spark_min, max as spark_max,
    regexp_extract, lit, lower, col
)
from pyspark.sql.window import Window

df_expanded = spark.table(TABLE_NAME)
print(f"Loaded {df_expanded.count():,} messages")

df_analysis = df_expanded.withColumn(
    "timestamp", to_timestamp(col("DateTimeSent"))
).withColumn(
    "date", date_format(col("timestamp"), "yyyy-MM-dd")
).withColumn(
    "hour", hour(col("timestamp"))
).withColumn(
    "day_of_week", dayofweek(col("timestamp"))
).withColumn(
    "message_length", length(col("Text"))
).withColumn(
    "sender_type", when(col("From").contains("@"), "Bot").otherwise("User")
)

df_filtered = df_analysis.filter(
    ~lower(col("Text")).contains("session has expired") &
    ~lower(col("Text")).contains("session expired") &
    ~lower(col("Text")).contains("session canceled") &
    ~lower(col("Text")).contains("session has been canceled")
)

# Note: .cache() is not supported on serverless compute.
# The DataFrames df_analysis and df_filtered are still reused across ~15 cells in 01_bi_analytics_dashboard.py.

convo_count = df_analysis.select("ConversationId").distinct().count()
msg_count = df_analysis.count()

print(f"Total conversations: {convo_count:,}")
print(f"Total messages: {msg_count:,}")
print(f"Avg messages per conversation: {msg_count / convo_count:.1f}")
print("\nDataFrames df_analysis and df_filtered are cached and ready for 01_bi_analytics_dashboard.py")
